In [ ]:
import os
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import torch
from ultralytics import YOLO
from IPython.display import Video, display

# =========================
# CONFIG
# =========================
VIDEO_PATH = "data/videos/58187_001383_Sideline.mp4"
PLAYER_MODEL_WEIGHTS = "runs/detect/football_positions3/weights/best.pt"
FIELD_MODEL_WEIGHTS  = "runs/detect/football_yolo11n2/weights/best.pt"

PRE_SNAP_SECONDS            = 5.0    # first 5 seconds = pre-snap
CONF_THRESHOLD_FOR_VOTE     = 0.5    # only count predictions with conf >= this
MIN_VOTES_PER_TRACK_FREEZE  = 8      # min high-conf votes for a track to get frozen
IOU_MATCH_THRESHOLD         = 0.3    # post-snap: IoU threshold to match detection to frozen player

device = 0 if torch.cuda.is_available() else "cpu"

# =========================
# Helper: IoU
# =========================
def iou(boxA, boxB):
    # box: (x1, y1, x2, y2)
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH
    if interArea <= 0:
        return 0.0

    boxAArea = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    boxBArea = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
    union = boxAArea + boxBArea - interArea
    if union <= 0:
        return 0.0
    return interArea / union

# =========================
# Load models
# =========================
player_model = YOLO(PLAYER_MODEL_WEIGHTS)
field_model  = YOLO(FIELD_MODEL_WEIGHTS)

player_names = player_model.names   # should be something like ['QB','SKILL','DB','LB','C']
field_names  = field_model.names

print("Player classes:", player_names)
print("Field classes:", field_names)

def class_id_to_role(cls_id: int) -> str:
    return player_names[cls_id]

# =========================
# Video properties
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH}")

fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

pre_snap_frames = int(round(PRE_SNAP_SECONDS * fps))
print(f"FPS: {fps:.2f}, pre-snap frames: {pre_snap_frames}")

# =========================
# Output writer
# =========================
os.makedirs("annotated_videos", exist_ok=True)
video_stem = Path(VIDEO_PATH).stem
out_path = Path("annotated_videos") / f"{video_stem}_annotated.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))

# =========================
# Role freezing state
# =========================
position_counts = defaultdict(Counter)  # track_id -> Counter(role -> count)
vote_counts     = defaultdict(int)      # track_id -> number of high-conf votes
frozen_players  = []                    # list of dicts: {'role', 'bbox'}

frozen = False  # becomes True after we finalize frozen_players

# =========================
# Colors per role (BGR)
# =========================
position_color_map = {
    "QB":    (0,   0, 255),   # red
    "SKILL": (0, 255,   0),   # green
    "DB":    (255, 0,   0),   # blue
    "LB":    (0, 255, 255),   # yellow
    "C":     (255, 0, 255),   # magenta
}
field_color = (255, 255, 0)

def get_color_for_position(pos_name: str):
    return position_color_map.get(pos_name, (0, 255, 0))

# =========================
# Tracking loop
# =========================
results_gen = player_model.track(
    source=VIDEO_PATH,
    imgsz=1280,
    conf=0.25,
    device=device,
    tracker="bytetrack.yaml",
    stream=True,
    persist=True,
)

frame_idx = 0
for r in results_gen:
    frame = r.orig_img.copy()  # BGR frame from video

    # 1) Field detections (optional)
    f_res = field_model(frame)[0]
    for box in f_res.boxes:
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        label = f"{field_names[cls]} {conf:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), field_color, 2)
        cv2.putText(frame, label, (x1, max(0, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, field_color, 2)

    boxes = r.boxes

    # 2) PRE-SNAP: use track IDs to accumulate votes
    if not frozen and frame_idx < pre_snap_frames and boxes is not None:
        for box in boxes:
            if box.id is None:
                continue
            tid  = int(box.id[0])
            cls  = int(box.cls[0])
            conf = float(box.conf[0])
            role = class_id_to_role(cls)

            if conf >= CONF_THRESHOLD_FOR_VOTE:
                position_counts[tid][role] += 1
                vote_counts[tid] += 1

    # When we reach the snap frame the first time, freeze roles
    if (not frozen) and frame_idx >= pre_snap_frames:
        for tid, counts in position_counts.items():
            if vote_counts[tid] >= MIN_VOTES_PER_TRACK_FREEZE and counts:
                frozen_role = counts.most_common(1)[0][0]
                # Take the last bbox we saw for this tid (approximate)
                # We can get it from the last frame where it appeared; here we’ll just use current frame if present.
                # To be safer, we reconstruct from current boxes if the id is present, else skip bbox init for now.
                if boxes is not None:
                    for box in boxes:
                        if box.id is not None and int(box.id[0]) == tid:
                            x1, y1, x2, y2 = map(int, box.xyxy[0])
                            frozen_players.append({
                                "tid": tid,
                                "role": frozen_role,
                                "bbox": (x1, y1, x2, y2)
                            })
                            break
        frozen = True
        print(
            f"Snap at frame {frame_idx}: created {len(frozen_players)} frozen players "
            f"(min votes {MIN_VOTES_PER_TRACK_FREEZE}, conf >= {CONF_THRESHOLD_FOR_VOTE})"
        )

    # 3) POST-SNAP (and also drawing during pre-snap)
    if boxes is not None:
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls  = int(box.cls[0])
            conf = float(box.conf[0])
            role_live = class_id_to_role(cls)
            det_box = (x1, y1, x2, y2)

            if frozen and frozen_players:
                # Match this detection to a frozen player by IoU
                best_iou = 0.0
                best_idx = None
                for i, fp in enumerate(frozen_players):
                    iou_val = iou(det_box, fp["bbox"])
                    if iou_val > best_iou:
                        best_iou = iou_val
                        best_idx = i
                if best_idx is not None and best_iou >= IOU_MATCH_THRESHOLD:
                    # Use frozen role, update bbox to keep track location fresh
                    frozen_players[best_idx]["bbox"] = det_box
                    role_disp = frozen_players[best_idx]["role"]
                else:
                    # Detection that doesn't match any frozen player (new or irrelevant)
                    role_disp = role_live
            else:
                # Pre-snap (or before we froze anything): use live prediction or majority so far, if you want
                # Here we’ll just show live prediction
                role_disp = role_live

            color = get_color_for_position(role_disp)
            label = f"{role_disp} {conf:.2f}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, max(0, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    writer.write(frame)
    frame_idx += 1

writer.release()
print(f"Saved annotated video to: {os.path.abspath(out_path)}")

display(Video(str(out_path), embed=True, width=800))
